In [ ]:
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Style configuration
plt.style.use('default')
plt.rcParams.update({
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.autolayout': True,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11
})

# Modern color palette
colors = ['#1a6fdf', '#37b6bd', '#ff6b6b', '#6b66ff', '#20c997']

# Widget configuration with improved spacing
slider_layout = widgets.Layout(width='500px', padding='0 0 15px 0')
label_style = {'description_width': '200px'}

controls = {
    'population': widgets.IntSlider(600, 400, 800, 50,
                                  description='<b>Population (millions):</b>',
                                  style=label_style, layout=slider_layout),
    'lambda_coeff': widgets.IntSlider(10, 5, 100, 5,
                                    description='<b>λ workflow multiplier:</b>',
                                    style=label_style, layout=slider_layout),
    'ia_growth': widgets.IntSlider(16, 10, 30, 1,
                                 description='<b>AI growth rate (%/yr):</b>',
                                 style=label_style, layout=slider_layout),
    'energy_efficiency': widgets.IntSlider(40, 0, 60, 5,
                                         description='<b>Energy efficiency (%):</b>',
                                         style=label_style, layout=slider_layout),
    'bandwidth_efficiency': widgets.IntSlider(50, 0, 70, 5,
                                            description='<b>Bandwidth efficiency (%):</b>',
                                            style=label_style, layout=slider_layout),
    'infra_loss': widgets.IntSlider(30, 0, 40, 5,
                                  description='<b>Infrastructure loss (%):</b>',
                                  style=label_style, layout=slider_layout),
    'edge_ai': widgets.IntSlider(20, 0, 50, 5,
                               description='<b>Edge AI adoption (%):</b>',
                               style=label_style, layout=slider_layout),
    'slm_adoption': widgets.IntSlider(30, 0, 60, 5,
                                    description='<b>SLM adoption (%):</b>',
                                    style=label_style, layout=slider_layout)
}

graph_output = widgets.Output()

def update_dashboard(change):
    with graph_output:
        clear_output(wait=True)

        # Time range (2024-2030)
        t = np.arange(0, 7)
        years = 2024 + t

        # Core calculations with proper scaling
        pop = controls['population'].value * 1e6  # Convert to absolute number
        daily_requests = 100 * controls['lambda_coeff'].value * (1 + controls['ia_growth'].value/100)**t

        # Energy calculations (GWh/day)
        energy_per_request = 0.005  # kWh per request (5 Wh)
        energy_base = pop * daily_requests * energy_per_request / 1e6  # Convert to GWh
        energy_base *= (1 + controls['infra_loss'].value/100)
        energy_opt = energy_base * (1 - controls['energy_efficiency'].value/100)

        # Bandwidth calculations (EB/day)
        bandwidth_per_request = 0.0035  # GB per request (3.5 MB)
        bw_base = pop * daily_requests * bandwidth_per_request / 1e6  # Convert to EB
        bw_opt = bw_base * (1 - controls['bandwidth_efficiency'].value/100)

        # CO2 calculations (kt/year)
        co2_per_gwh = 500  # kg CO2 per GWh (renewable energy mix)
        co2_base = energy_base * 365 * co2_per_gwh / 1e6  # Convert to kt CO2/year
        co2_opt = co2_base * (1 - max(controls['energy_efficiency'].value, controls['bandwidth_efficiency'].value)/100)

        # Create figure grid
        fig = plt.figure(figsize=(18, 18))
        gs = fig.add_gridspec(3, 2)

        # 1. Daily Energy Consumption
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.plot(years, energy_base, color=colors[1], lw=3, label='Standard')
        ax1.plot(years, energy_opt, color=colors[2], lw=3, label='Optimized')
        ax1.set_title('DAILY ENERGY CONSUMPTION', pad=20)
        ax1.set_ylabel('GWh/day', labelpad=10)
        ax1.legend()

        # 2. Daily Bandwidth Consumption
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.plot(years, bw_base, color=colors[1], lw=3, label='Standard')
        ax2.plot(years, bw_opt, color=colors[2], lw=3, label='Optimized')
        ax2.set_title('DAILY BANDWIDTH CONSUMPTION', pad=20)
        ax2.set_ylabel('EB/day', labelpad=10)
        ax2.legend()

        # 3. AI Workflow Multiplier Effect
        ax3 = fig.add_subplot(gs[1, 0])
        λ_values = np.linspace(5, 100, 20)
        ax3.plot(λ_values, 100*λ_values, color=colors[0], lw=3)
        ax3.axvline(controls['lambda_coeff'].value, color='red', linestyle='--')
        ax3.set_title('AI WORKFLOW MULTIPLIER EFFECT', pad=20)
        ax3.set_xlabel('λ coefficient', labelpad=10)
        ax3.set_ylabel('Effective requests/user/day', labelpad=10)

        # 4. Annual Carbon Footprint
        ax4 = fig.add_subplot(gs[1, 1])
        ax4.plot(years, co2_base, color=colors[1], lw=3, label='Standard')
        ax4.plot(years, co2_opt, color=colors[2], lw=3, label='Optimized')
        ax4.set_title('ANNUAL CARBON FOOTPRINT', pad=20)
        ax4.set_ylabel('kt CO₂/year', labelpad=10)
        ax4.legend()

        # 5. Scenario Comparison (2030)
        ax5 = fig.add_subplot(gs[2, 0])
        scenarios = ['Web', 'Standard AI', 'Optimized AI']
        web_energy, web_bw = 1.2, 0.03
        metrics = {
            'Energy (GWh/d)': [web_energy, energy_base[-1], energy_opt[-1]],
            'Bandwidth (EB/d)': [web_bw, bw_base[-1], bw_opt[-1]]
        }
        x = np.arange(len(scenarios))
        width = 0.35
        for i, (metric, values) in enumerate(metrics.items()):
            ax5.bar(x + i*width, values, width, label=metric,
                   color=colors[i*2])
        ax5.set_title('SCENARIO COMPARISON IN 2030', pad=20)
        ax5.set_xticks(x + width/2)
        ax5.set_xticklabels(scenarios)
        ax5.legend()

        # 6. Sustainable Solutions Adoption
        ax6 = fig.add_subplot(gs[2, 1])
        solutions = ['Edge AI', 'SLMs']
        adoption = [controls['edge_ai'].value, controls['slm_adoption'].value]
        ax6.barh(solutions, adoption, color=[colors[4], colors[3]])
        ax6.set_title('SUSTAINABLE SOLUTIONS ADOPTION', pad=20)
        ax6.set_xlabel('Adoption Rate (%)', labelpad=10)
        ax6.set_xlim(0, 100)

        plt.tight_layout()
        plt.show()

        # Key metrics display
        print(f"\n🔍 2030 PROJECTIONS (λ={controls['lambda_coeff'].value})")
        print(f"• Energy: {energy_base[-1]:.0f} → {energy_opt[-1]:.0f} GWh/day")
        print(f"• Bandwidth: {bw_base[-1]:.1f} → {bw_opt[-1]:.1f} EB/day")
        print(f"• CO₂ Emissions: {co2_base[-1]:.1f} → {co2_opt[-1]:.1f} kt/year")

# Link controls to update
for control in controls.values():
    control.observe(update_dashboard, 'value')

# Interface organization with improved spacing
left_panel = widgets.VBox([
    widgets.Label("DEMOGRAPHIC PARAMETERS"),
    controls['population'],
    widgets.Label("AI WORKFLOW PARAMETERS"),
    controls['lambda_coeff'],
    controls['ia_growth']
], layout=widgets.Layout(padding='0 20px 0 0'))

right_panel = widgets.VBox([
    widgets.Label("OPTIMIZATION PARAMETERS"),
    controls['energy_efficiency'],
    controls['bandwidth_efficiency'],
    controls['infra_loss'],
    widgets.Label("SUSTAINABLE SOLUTIONS"),
    controls['edge_ai'],
    controls['slm_adoption']
], layout=widgets.Layout(padding='0 0 0 20px'))

# Vertical spacer
spacer = widgets.Label("", layout=widgets.Layout(height='15px'))

dashboard = widgets.HBox([
    left_panel,
    spacer,
    right_panel
], layout=widgets.Layout(justify_content='space-between'))

# Initial display
display(dashboard)
display(graph_output)
update_dashboard(None)

Output()